## Consolidación de base_clean.py v1

Junto en un solo lugar todo lo que fui confirmando en el diagnóstico: la
función de carga con el fix de columnas de `devices`, las funciones de
diagnóstico (pandas para tablas chicas, SQL para `transactions` por su
volumen), los chequeos puntuales que usé para investigar cada hallazgo, y
finalmente las funciones `clean_*` por tabla con los filtros ya decididos
y justificados:

- `clean_devices`: corrige el renombre de columnas y elimina la fila de
  header que se coló como dato
- `clean_users`: descarta `num_referrals`/`num_successful_referrals` (cero
  varianza, sin poder predictivo), calcula `age` con `REFERENCE_DATE` fija,
  y aplica el filtro de edad plausible [18, 122] como salvaguarda aunque
  hoy no elimine ninguna fila
- `clean_notifications`: pasa sin cambios, no encontré nada que limpiar
- `clean_transactions`: filtra montos ≤0 y el bug de conversión de moneda
  (>$1M, confinado a `DECLINED`), y agrego `has_merchant_info` para no
  confundir los nulls estructurales de `ea_*` con datos faltantes reales

Dejo pendientes dos decisiones que quiero resolver contigo antes de congelar
esta versión: cómo tratar los nulls de `marketing_push`/`marketing_email`
(¿imputar como "unknown"?) y los nulls de `ea_*` (¿NaN explícito o
"NOT_APPLICABLE"?).

In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# Configuración general
# ---------------------------------------------------------------------------
PROJECT_ID = "quiet-seer-299408"
DATASET = "neobank"

client = bigquery.Client(project=PROJECT_ID)

# Fecha de referencia FIJA para todos los cálculos de edad/antigüedad.
REFERENCE_DATE = pd.Timestamp("2026-07-01")

MIN_AGE, MAX_AGE = 18, 122  # récord de longevidad verificado (Jeanne Calment, GRG)
CARD_RELATED_TYPES = ["CARD_PAYMENT", "ATM", "CARD_REFUND"]
AMOUNT_USD_UPPER_BOUND = 1_000_000  # bug de conversión confirmado, confinado a DECLINED


# ---------------------------------------------------------------------------
# Funciones de carga
# ---------------------------------------------------------------------------

def load_table(table_name: str) -> pd.DataFrame:
    """Carga una tabla completa del dataset neobank como DataFrame."""
    query = f"SELECT * FROM `{PROJECT_ID}.{DATASET}.{table_name}`"
    return client.query(query).to_dataframe()


def load_devices() -> pd.DataFrame:
    """
    Carga `devices` renombrando las columnas mal inferidas por BigQuery.
    string_field_0 -> device_brand (confirmado por muestra: 'Apple', etc.)
    string_field_1 -> user_id
    """
    query = f"""
        SELECT
            string_field_1 AS user_id,
            string_field_0 AS device_brand
        FROM `{PROJECT_ID}.{DATASET}.devices`
    """
    return client.query(query).to_dataframe()


# ---------------------------------------------------------------------------
# Funciones de limpieza por tabla
# ---------------------------------------------------------------------------

def clean_users() -> pd.DataFrame:
    """
    - Descarta num_referrals y num_successful_referrals (varianza cero)
    - Calcula age a partir de birth_year y REFERENCE_DATE (no CURRENT_DATE)
    - Aplica el filtro de edad plausible [18, 122]
    - Imputa nulls de marketing_push/email como 'unknown' (cohorte 2018-2019),
      convirtiendo primero 0.0/1.0 -> "0"/"1" para que la columna quede
      100% string y no rompa la exportación a Parquet
    """
    df = load_table("users")

    df["age"] = REFERENCE_DATE.year - df["birth_year"]
    df = df[(df["age"] >= MIN_AGE) & (df["age"] <= MAX_AGE)].reset_index(drop=True)

    df = df.drop(columns=["num_referrals", "num_successful_referrals"])

    for col in ["attributes_notifications_marketing_push",
                "attributes_notifications_marketing_email"]:
        df[col] = df[col].astype("Int64").astype("string")
        df[col] = df[col].fillna("unknown")

    return df


def clean_devices() -> pd.DataFrame:
    """
    - Renombra string_field_0/1 -> device_brand/user_id
    - Elimina la fila de header mal cargada (user_id == 'user_id')
    """
    df = load_devices()
    df = df[df["user_id"] != "user_id"].reset_index(drop=True)
    return df


def clean_notifications() -> pd.DataFrame:
    """Sin hallazgos de calidad que requieran filtro (0 nulls, 0 duplicados,
    0 user_id huérfanos frente a users)."""
    return load_table("notifications")


def clean_transactions() -> pd.DataFrame:
    """
    - Filtra amount_usd <= 0 y > 1,000,000 (bug de conversión, confinado a DECLINED)
    - Marca has_merchant_info según transactions_type
    - Imputa nulls estructurales de merchant (ea_*) como 'NOT_APPLICABLE',
      convirtiendo ea_merchant_mcc a string antes (es FLOAT en origen, y
      mezclarlo con el string 'NOT_APPLICABLE' rompe la exportación a Parquet)
    """
    df = load_table("transactions")

    df = df[
        (df["amount_usd"] > 0) & (df["amount_usd"] <= AMOUNT_USD_UPPER_BOUND)
    ].reset_index(drop=True)

    df["has_merchant_info"] = df["transactions_type"].isin(CARD_RELATED_TYPES)

    # ea_merchant_mcc es FLOAT en origen (ej. 5411.0) -> lo paso a string
    # (sin decimales) antes de imputar, para no mezclar float + string
    df["ea_merchant_mcc"] = df["ea_merchant_mcc"].astype("Int64").astype("string")

    merchant_cols = ["ea_cardholderpresence", "ea_merchant_mcc",
                      "ea_merchant_city", "ea_merchant_country"]
    for col in merchant_cols:
        df[col] = df[col].astype("string")
        df[col] = df[col].fillna("NOT_APPLICABLE")

    return df

In [2]:
clean_users_df = clean_users()
clean_devices_df = clean_devices()
clean_notifications_df = clean_notifications()
clean_transactions_df = clean_transactions()

print("users:", clean_users_df.shape)
print("devices:", clean_devices_df.shape)
print("notifications:", clean_notifications_df.shape)
print("transactions:", clean_transactions_df.shape)

print("\n--- Verificando que ya no hay nulls ---")
print(clean_transactions_df[["ea_cardholderpresence", "ea_merchant_mcc", "ea_merchant_city", "ea_merchant_country"]].isna().sum())
print(clean_users_df[["attributes_notifications_marketing_push", "attributes_notifications_marketing_email"]].isna().sum())

users: (19430, 11)
devices: (19430, 2)
notifications: (121813, 5)
transactions: (2681355, 13)

--- Verificando que ya no hay nulls ---
ea_cardholderpresence    0
ea_merchant_mcc          0
ea_merchant_city         0
ea_merchant_country      0
dtype: int64
attributes_notifications_marketing_push     0
attributes_notifications_marketing_email    0
dtype: int64


In [3]:
clean_users_df.to_parquet("../data/users_clean.parquet", index=False)
clean_devices_df.to_parquet("../data/devices_clean.parquet", index=False)
clean_notifications_df.to_parquet("../data/notifications_clean.parquet", index=False)
clean_transactions_df.to_parquet("../data/transactions_clean.parquet", index=False)